In [ ]:
import pandas as pd
import pymc as pm
import matplotlib.pyplot as plt
import numpy as np
import arviz as az

np.random.seed(123)

data = pd.read_csv("https://github.com/dustywhite7/Econ8310/raw/master/AssignmentData/cookie_cats.csv")

data.head()


,userid,version,sum_gamerounds,retention_1,retention_7
0,116,gate_30,3,False,False
1,337,gate_30,38,True,False
2,377,gate_40,165,True,False
3,483,gate_40,1,False,False
4,488,gate_40,179,True,True


In [ ]:
data["version"].value_counts()

,count
version,
gate_40,45489
gate_30,44700


In [ ]:
with pm.Model() as model_1day:

    # Priors (our starting belief)
    p_A = pm.Beta("p_A", alpha=1, beta=1)
    p_B = pm.Beta("p_B", alpha=1, beta=1)

    # Difference between groups
    delta = pm.Deterministic("delta", p_A - p_B)

    # Observed data
    obs_A = pm.Bernoulli(
        "obs_A",
        p=p_A,
        observed=data.loc[data["version"] == "gate_30", "retention_1"]
    )

    obs_B = pm.Bernoulli(
        "obs_B",
        p=p_B,
        observed=data.loc[data["version"] == "gate_40", "retention_1"]
    )

    # Sampling (this runs MCMC)
    trace_1day = pm.sample(2000, tune=1000, cores=2)

Output()

In [ ]:
# Extract posterior samples
p_A_samples = trace_1day.posterior["p_A"].values.flatten()
p_B_samples = trace_1day.posterior["p_B"].values.flatten()
delta_samples = trace_1day.posterior["delta"].values.flatten()

In [ ]:
# Probability that gate_30 is better than gate_40
prob = (delta_samples > 0).mean()

print("Probability gate_30 > gate_40:", prob)

# Average effect
print("Average difference (p_A - p_B):", delta_samples.mean())

Probability gate_30 > gate_40: 0.9615
Average difference (p_A - p_B): 0.005868844351571346


In [ ]:
with pm.Model() as model_7day:

    p_A = pm.Beta("p_A_7", alpha=1, beta=1)
    p_B = pm.Beta("p_B_7", alpha=1, beta=1)

    delta = pm.Deterministic("delta_7", p_A - p_B)

    obs_A = pm.Bernoulli(
        "obs_A_7",
        p=p_A,
        observed=data.loc[data["version"] == "gate_30", "retention_7"]
    )

    obs_B = pm.Bernoulli(
        "obs_B_7",
        p=p_B,
        observed=data.loc[data["version"] == "gate_40", "retention_7"]
    )

    trace_7day = pm.sample(2000, tune=1000, cores=2)

Output()

In [ ]:
p_A_samples_7day = trace_7day.posterior["p_A_7"].values.flatten()
p_B_samples_7day = trace_7day.posterior["p_B_7"].values.flatten()
delta_samples_7day = trace_7day.posterior["delta_7"].values.flatten()

prob_7day = (delta_samples_7day > 0).mean()

print("Probability gate_30 > gate_40:", prob_7day)
print("Average difference (p_A - p_B):", delta_samples_7day.mean())

Probability gate_30 > gate_40: 0.99925
Average difference (p_A - p_B): 0.008165564632556624
